## Import Polars

In [1]:
%uv add polars

/workspaces/endjin-polars-examples/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [2]:
import polars as pl

## Read Data into DataFrame

In [3]:
COLUMN_NAMES = [
        "id", "price", "date", "postcode", "property_type",
        "old_new", "duration", "paon", "saon", "street",
        "locality", "town_city", "district", "county",
        "ppd_category", "record_type",
    ]

In [4]:
land_registry_data = pl.read_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""],
)

In [5]:
land_registry_data

id,price,date,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_type
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""{D707E535-5720-0AD9-E053-6B04A…",260000,"""2021-08-06 00:00""","""SO45 2HT""","""T""","""N""","""F""","""17""",null,"""PERRYWOOD CLOSE""","""HOLBURY""","""SOUTHAMPTON""","""NEW FOREST""","""HAMPSHIRE""","""A""","""A"""
"""{D707E535-5721-0AD9-E053-6B04A…",375000,"""2021-09-01 00:00""","""SO23 7FR""","""S""","""N""","""F""","""1""",null,"""BOXALL GARDENS""","""KINGS WORTHY""","""WINCHESTER""","""WINCHESTER""","""HAMPSHIRE""","""A""","""A"""
"""{D707E535-5723-0AD9-E053-6B04A…",132000,"""2021-06-28 00:00""","""SP11 6RL""","""F""","""N""","""L""","""2""",null,"""SEDGE ROAD""",null,"""ANDOVER""","""TEST VALLEY""","""HAMPSHIRE""","""A""","""A"""
"""{D707E535-5724-0AD9-E053-6B04A…",295000,"""2021-09-10 00:00""","""SP11 6TU""","""T""","""N""","""F""","""21""",null,"""NAP CLOSE""",null,"""ANDOVER""","""TEST VALLEY""","""HAMPSHIRE""","""A""","""A"""
"""{D707E535-5725-0AD9-E053-6B04A…",360000,"""2021-08-27 00:00""","""SO51 0AX""","""T""","""N""","""F""","""130""",null,"""FREEMANTLE ROAD""",null,"""ROMSEY""","""TEST VALLEY""","""HAMPSHIRE""","""A""","""A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""{36A61A95-56B0-DEF2-E063-4704A…",177500,"""2025-04-30 00:00""","""NR1 3SB""","""F""","""N""","""L""","""128""",null,"""BRAZEN GATE""",null,"""NORWICH""","""NORWICH""","""NORFOLK""","""B""","""A"""
"""{36A61A95-56B2-DEF2-E063-4704A…",113501,"""2025-05-02 00:00""","""PE30 5HJ""","""T""","""N""","""F""","""24""",null,"""SOUTH EVERARD STREET""",null,"""KING'S LYNN""","""KING'S LYNN AND WEST NORFOLK""","""NORFOLK""","""B""","""A"""
"""{36A61A95-56B5-DEF2-E063-4704A…",1100000,"""2025-04-24 00:00""","""NR16 1QH""","""O""","""N""","""F""","""HAY BARN HOUSE""",null,"""CARGATE COMMON""","""TIBENHAM""","""NORWICH""","""SOUTH NORFOLK""","""NORFOLK""","""B""","""A"""


In [6]:
land_registry_data = (
    land_registry_data
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
)

In [7]:
land_registry_data["property_type"].value_counts()

property_type,count
str,u32
"""D""",1135907
"""F""",878918
"""T""",1344870
"""S""",1317496
"""O""",264930


In [8]:
land_registry_data = land_registry_data.with_columns(
    pl.col("property_type").replace({
        "D": "Detached",
        "S": "Semi-Detached",
        "T": "Terraced",
        "F": "Flat",
        "O": "Other",
    }).alias("property_type")
)

In [9]:
land_registry_data["property_type"].value_counts()

property_type,count
str,u32
"""Flat""",878918
"""Detached""",1135907
"""Semi-Detached""",1317496
"""Other""",264930
"""Terraced""",1344870


In [10]:
land_registry_data = land_registry_data.filter(pl.col("property_type") != "Other")

In [11]:
land_registry_data = (
    land_registry_data
    .with_columns(
        pl.col("date").dt.year().alias("year"),
    )
)

In [12]:
land_registry_data.select(["id", "date", "year", "property_type", "price"])

id,date,year,property_type,price
str,date,i32,str,i64
"""{D707E535-5720-0AD9-E053-6B04A…",2021-08-06,2021,"""Terraced""",260000
"""{D707E535-5721-0AD9-E053-6B04A…",2021-09-01,2021,"""Semi-Detached""",375000
"""{D707E535-5723-0AD9-E053-6B04A…",2021-06-28,2021,"""Flat""",132000
"""{D707E535-5724-0AD9-E053-6B04A…",2021-09-10,2021,"""Terraced""",295000
"""{D707E535-5725-0AD9-E053-6B04A…",2021-08-27,2021,"""Terraced""",360000
…,…,…,…,…
"""{36A61A95-56AD-DEF2-E063-4704A…",2025-05-22,2025,"""Semi-Detached""",267500
"""{36A61A95-56AE-DEF2-E063-4704A…",2025-04-29,2025,"""Semi-Detached""",230000
"""{36A61A95-56B0-DEF2-E063-4704A…",2025-04-30,2025,"""Flat""",177500


In [13]:
annual_price_by_property_type = (
    land_registry_data
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

In [14]:
annual_price_by_property_type

year,property_type,median_price
i32,str,f64
2021,"""Detached""",386950.0
2021,"""Flat""",230000.0
2021,"""Semi-Detached""",244995.0
2021,"""Terraced""",210000.0
2022,"""Detached""",420000.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0


## Visualise the results

In [15]:
import plotly.express as px

In [16]:
px.line(
    annual_price_by_property_type,
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

In [17]:
annual_price_by_property_type.write_delta("../../data/price_paid_insights/annual_price_by_property_type", mode="overwrite")

In [18]:
(
    pl.read_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""])
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
    .with_columns(
        pl.col("property_type").replace({
            "D": "Detached",
            "S": "Semi-Detached",
            "T": "Terraced",
            "F": "Flat",
            "O": "Other",
        }).alias("property_type")
    )
    .filter(pl.col("property_type") != "Other")
    .with_columns(
        pl.col("date").dt.year().alias("year")
    )
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

year,property_type,median_price
i32,str,f64
2021,"""Detached""",386950.0
2021,"""Flat""",230000.0
2021,"""Semi-Detached""",244995.0
2021,"""Terraced""",210000.0
2022,"""Detached""",420000.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0


In [19]:
lazy_frame = (
    pl.scan_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""])
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
    .with_columns(
        pl.col("property_type").replace({
            "D": "Detached",
            "S": "Semi-Detached",
            "T": "Terraced",
            "F": "Flat",
            "O": "Other",
        }).alias("property_type")
    )
    .filter(pl.col("property_type") != "Other")
    .with_columns(
        pl.col("date").dt.year().alias("year")
    )
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

In [20]:
print(lazy_frame.explain(optimized=True))

SORT BY [col("year"), col("property_type")]
  AGGREGATE[maintain_order: false]
    [col("price").median().alias("median_price")] BY [col("year"), col("property_type")]
    FROM
     WITH_COLUMNS:
     [col("date").dt.year().alias("year")] 
      FILTER [(col("property_type")) != ("Other")]
      FROM
         WITH_COLUMNS:
         [col("date").str.strptime(["raise"]), col("property_type").replace([["D", "S", … "O"], ["Detached", "Semi-Detached", … "Other"]])] 
          Csv SCAN [../../data/land_registry_data/pp-2021.csv, ... 4 other sources]
          PROJECT 3/16 COLUMNS
          ESTIMATED ROWS: 1251169


In [21]:
lazy_frame.collect()

year,property_type,median_price
i32,str,f64
2021,"""Detached""",386950.0
2021,"""Flat""",230000.0
2021,"""Semi-Detached""",244995.0
2021,"""Terraced""",210000.0
2022,"""Detached""",420000.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0
